In [1]:
from gnncloudmanufacturing.data import read_fatahi_dataset
from gnncloudmanufacturing.random_solver import random_solve
from gnncloudmanufacturing.validation import total_cost_from_graph, check_feasibility, total_cost_from_gamma
from gnncloudmanufacturing.utils import delta_from_gamma, graph_from_problem, gamma_from_target, delta_from_gamma
from gnncloudmanufacturing.graph_model import GNN, os_type, ss_type

import numpy as np
from tqdm.auto import trange, tqdm
from time import time
import pandas as pd
import torch

In [2]:
dataset = read_fatahi_dataset('../../data/fatahi.xlsx', sheet_names=['5,5,5-1', '5,5,5-2', '5,5,5-3'])
for problem in dataset:
    print(f'Problem: {problem["name"]}')

  0%|          | 0/3 [00:00<?, ?it/s]

Problem: 5,5,5-1
Problem: 5,5,5-2
Problem: 5,5,5-3


In [3]:
max_operations = 5
model = GNN.load_from_checkpoint(
    checkpoint_path="gnn-5-5-5.ckpt",
    ins_dim=1,
    ino_dim=max_operations,
    out_dim=16,
    n_layers=1,
    lr=0.002,
)
model.eval()

GNN(
  (convs): ModuleList(
    (0): AttnConvLayer(
      (W_s): Linear(in_features=1, out_features=16, bias=True)
      (W_os): Linear(in_features=7, out_features=16, bias=True)
      (W_ss): Linear(in_features=2, out_features=16, bias=True)
      (attn): Linear(in_features=32, out_features=1, bias=True)
      (W_in): Linear(in_features=5, out_features=16, bias=True)
      (W_self): Linear(in_features=5, out_features=16, bias=True)
      (W_out): Linear(in_features=5, out_features=16, bias=True)
      (W_o): Linear(in_features=48, out_features=16, bias=True)
    )
  )
  (dropout): Dropout(p=0.0, inplace=False)
  (dec): DotProductDecoder()
)

In [4]:
problem_name = []
total_cost = []
comp_time = []
for problem in tqdm(dataset):
    start = time()
    total = np.inf
    for i in range(5):
        graph = graph_from_problem(problem, max_operations=max_operations)
        graph.edata['feat'][os_type][:, 0] /= 10
        graph.edata['feat'][ss_type][:] /= 100
        pred = model.predict(graph)
        gamma = gamma_from_target(pred, graph, problem)
        delta = delta_from_gamma(problem, gamma)
        check_feasibility(gamma, delta, problem)
        _total = total_cost_from_gamma(problem, gamma, delta).item()
        if total > _total:
            total = _total
    total_cost.append(total)
    problem_name.append(problem['name'])
    comp_time.append(time() - start)

  0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
pd.DataFrame({'problem_name': problem_name, 'total_cost': total_cost, 'comp_time': comp_time}).round(2)

,problem_name,total_cost,comp_time
0,"5,5,5-1",3247.22,0.09
1,"5,5,5-2",5944.42,0.03
2,"5,5,5-3",6653.88,0.03
